<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Cube_3D_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cube 3D + CubePart — Roblox 3D Generation Suite

Two complementary 3D pipelines from [Roblox's Cube project](https://github.com/Roblox/cube), open-sourced as part of their push toward 3D intelligence.

- **Cube 3D v0.5** (`Roblox/cube3d-v0.5`, July 2025) — autoregressive text-to-shape generation. Given a natural-language prompt (and a target bounding box), produces a single `.glb` 3D mesh. Uses a VQ-VAE tokenizer + a GPT-style shape transformer. ~12 GB of weights.
- **CubePart** (`Roblox/cubepart`, May 2026, SIGGRAPH '26) — open-vocabulary part-controllable **mesh decomposition** (not text-to-mesh). Given an existing canonically-aligned `.glb` mesh and a list of part names (e.g. `body, front left wheel, front right wheel, headlights`), produces a colored `trimesh.Scene` with one independent mesh per part. Two-stage DiT with cross-part attention. ~10 GB of weights. The upstream open-source release is Stage 2 (decomposition) only; Stage 1 (text → full mesh latent) is not yet released.

Both pipelines use the official Roblox code (cloned from `Roblox/cube` in Step 1). The Cube 3D v0.5 EngineFast backend is CUDA-only; CubePart runs on any GPU with ≥16 GB VRAM. The full notebook needs ~24 GB VRAM to load both engines simultaneously; otherwise free one with the buttons in the **VRAM** tab before loading the other.

**Disclaimer:** Generated assets are not moderated by Roblox safety systems. Use of the weights is subject to the [Cube repository license](https://github.com/Roblox/cube/blob/main/LICENSE). Outputs are yours to use in your own game engines, physics rendering, and animation pipelines.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones the [Roblox/cube](https://github.com/Roblox/cube) repo, installs Cube 3D + CubePart
# @markdown and Gradio. The first run takes ~3-5 minutes (git clone + pip install). Subsequent runs are instant.

import os, sys, subprocess, importlib, pathlib, shutil

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime → Change runtime type → L4 or A100')

gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB — sm_{cap[0]}{cap[1]}')

if vram_gb < 16:
    raise SystemExit(
        f'❌ {vram_gb:.0f} GB VRAM is not enough. Cube 3D v0.5 EngineFast needs ~16 GB and CubePart needs ~20 GB. '
        'Reconnect to a runtime with ≥24 GB (L4 or A100).'
    )

REPO = 'cube'
REPO_DIR = pathlib.Path(f'/content/{REPO}')
REPO_URL = 'https://github.com/Roblox/cube.git'

if not REPO_DIR.exists():
    print(f'[git] Cloning {REPO_URL} (~5-10 MB source) ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.')

print('[pip] Installing Cube 3D + CubePart dependencies ...')
subprocess.run(['pip', 'install', '-q', '-U',
               'huggingface_hub',
               'gradio>=4.40',
               'trimesh[easy]',
               'numpy<2.0',  # some upstream deps still expect numpy 1.x
               'pymeshlab',  # optional mesh simplification
               'rembg',  # optional background removal (future-proofing)
               'huggingface_hub[cli]',
               'tqdm'], check=True)  # tqdm is used by the upstream diffusion loop

print('[pip] Installing CubePart package (editable) ...')
subprocess.run(['pip', 'install', '-q', '-e', f'{REPO_DIR}/cubepart[demo]'], check=True)

# Cube 3D uses the parent 'cube' package, not a separate setup.py; add to sys.path manually.
CUBE3D_PKG = REPO_DIR / 'cube3d'
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if CUBE3D_PKG.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.makedirs('/content/cube3d_out', exist_ok=True)
os.makedirs('/content/ref_meshes', exist_ok=True)
print('[Setup] /content/cube3d_out and /content/ref_meshes ready.')

# Quick smoke import test
try:
    from cube3d.inference.engine import EngineFast
    print('[Import] cube3d.inference.engine.EngineFast OK')
except Exception as e:
    print(f'[Import] cube3d FAILED: {e}')

try:
    from cube_part.pipelines import PartShapeDenoiserPipeline
    print('[Import] cube_part.pipelines.PartShapeDenoiserPipeline OK')
except Exception as e:
    print(f'[Import] cube_part FAILED: {e}')

print('[Setup] All imports verified.')


In [ ]:
# @title STEP 2 — Pre-cache Model Weights
# @markdown Downloads the Cube 3D v0.5 + CubePart weights to Google Drive (~20-25 GB total).
# @markdown Reuses the Drive-cached weights from previous sessions if present.
# @markdown Otherwise downloads to the default ~/.cache/huggingface (in-session only).

import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')

from huggingface_hub import snapshot_download
import time

MODELS = [
    # (display_name, repo_id, local_subdir, file_patterns, size_est)
    ('Cube 3D v0.5 (GPT + shape tokenizer)', 'Roblox/cube3d-v0.5',
     'cube3d-v0.5', ['*.safetensors', '*.json', '*.yaml', '*.txt'], '~12 GB'),
    ('CubePart (multi-part DiT + shape VAE)', 'Roblox/cubepart',
     'cubepart', ['multi_part_dit.safetensors', 'vae.safetensors'], '~10 GB'),
]
for name, repo, subdir, patterns, size in MODELS:
    print(f'[Download] {name}  {repo}  ({size}) ...')
    t0 = time.time()
    try:
        snapshot_download(
            repo_id=repo,
            local_dir=f'/content/{subdir}',
            allow_patterns=patterns,
            max_workers=4,
        )
        print(f'  [Done] {time.time()-t0:.0f}s\n')
    except Exception as e:
        print(f'  [Fail] {e}\n')

print('[Cache] All selected weights cached. Local paths:')
print('  /content/cube3d-v0.5/shape_gpt.safetensors')
print('  /content/cube3d-v0.5/shape_tokenizer.safetensors')
print('  /content/cubepart/multi_part_dit.safetensors')
print('  /content/cubepart/vae.safetensors')

# Pre-fetch the 5 example .glb inputs that ship with the upstream CubePart repo
# (one drone, one flower, one jellyfish_car, one pirate_chest, one wizard).
# These are LFS-tracked on GitHub, so we use media.githubusercontent.com (which
# serves the actual binary content, not the LFS pointer text). The CubePart
# tab in Step 4 has a Gallery that lets the user pick one of these to test on
# without having to bring their own .glb file.
import urllib.request, shutil, os

EXAMPLE_MESHES = [
    # (name, url, expected_min_bytes, parts_pretty)
    ('drone',         'https://media.githubusercontent.com/media/Roblox/cube/main/cubepart/examples/inputs/drone.glb',         100_000, ('body', 'blades', 'landing legs')),
    ('flower',        'https://media.githubusercontent.com/media/Roblox/cube/main/cubepart/examples/inputs/flower.glb',        100_000, ('flower stem', 'flower petals', 'leaves')),
    ('jellyfish_car',  'https://media.githubusercontent.com/media/Roblox/cube/main/cubepart/examples/inputs/jellyfish_car.glb',  100_000, ('body', 'front right wheel', 'front left wheel', 'rear right wheel', 'rear left wheel', 'exhaust pipe', 'headlights', 'gun')),
    ('pirate_chest',   'https://media.githubusercontent.com/media/Roblox/cube/main/cubepart/examples/inputs/pirate_chest.glb',   100_000, ('lid', 'base')),
    ('wizard',        'https://media.githubusercontent.com/media/Roblox/cube/main/cubepart/examples/inputs/wizard.glb',        100_000, ('head', 'feather', 'hat', 'torso body', 'cloak', 'hands', 'magic staff', 'orb')),
]
MESH_DIR = Path('/content/ref_meshes')
MESH_DIR.mkdir(parents=True, exist_ok=True)
THUMB_DIR = MESH_DIR / 'thumbs'
THUMB_DIR.mkdir(parents=True, exist_ok=True)
print(f'\n[Mesh] Pre-fetching {len(EXAMPLE_MESHES)} example .glb inputs + PNG thumbnails into {MESH_DIR}/ ...')
for name, url, min_size, _ in EXAMPLE_MESHES:
    out_path = MESH_DIR / f'{name}.glb'
    if out_path.exists() and out_path.stat().st_size > min_size:
        print(f'  [Skip] {name}.glb (cached, {out_path.stat().st_size//1024} KB)')
    else:
        print(f'  [Fetch] {name}.glb from upstream ...')
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r, open(out_path, 'wb') as fh:
                shutil.copyfileobj(r, fh)
            print(f'    [Done] {out_path.stat().st_size//1024} KB')
        except Exception as e:
            print(f'    [Fail] {e}')
    # Thumbnail (used in the Examples gallery in Step 4)
    thumb_path = THUMB_DIR / f'{name}.png'
    if not thumb_path.exists() or thumb_path.stat().st_size < 1024:
        try:
            thumb_url = url.replace('/inputs/', '/inputs/').replace('.glb', '.png')
            req = urllib.request.Request(thumb_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=60) as r, open(thumb_path, 'wb') as fh:
                shutil.copyfileobj(r, fh)
        except Exception:
            pass  # thumbs are optional
print(f'[Mesh] Example meshes ready in {MESH_DIR}/')


In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers. Each engine loads on first call and caches in a global.
# @markdown The Cube 3D EngineFast is CUDA-only; the CubePart pipeline runs on any device with ≥16 GB VRAM.

import os, sys, time, json, tempfile, traceback, glob
from pathlib import Path
import numpy as np
import torch
import trimesh

REPO_DIR = Path('/content/cube')
CACHE_DIR = Path('/content')  # where Step 2 downloaded weights

CUBE3D_DIR = CACHE_DIR / 'cube3d-v0.5'
CUBE3D_GPT = CUBE3D_DIR / 'shape_gpt.safetensors'
CUBE3D_VAE = CUBE3D_DIR / 'shape_tokenizer.safetensors'
CUBE3D_CONFIG = None  # searched below

CUBEPART_DIR = CACHE_DIR / 'cubepart'
CUBEPART_DIT = CUBEPART_DIR / 'multi_part_dit.safetensors'
CUBEPART_VAE = CUBEPART_DIR / 'vae.safetensors'
CUBEPART_CONFIG = None  # searched below

OUT_DIR = Path('/content/cube3d_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-discover configs from the cloned repo
def _find_config(folder: Path, hint: str) -> Path | None:
    if not folder.exists():
        return None
    matches = list(folder.rglob('*.yaml'))
    for m in matches:
        if hint in m.name:
            return m
    return matches[0] if matches else None

CUBE3D_CONFIG = _find_config(REPO_DIR, 'v0.5') if REPO_DIR.exists() else None
CUBEPART_CONFIG = (
    REPO_DIR / 'cubepart' / 'configs' / 'shape_denoiser_multimesh.yaml'
    if (REPO_DIR / 'cubepart' / 'configs').exists() else None
)
print(f'[Config] Cube 3D: {CUBE3D_CONFIG}')
print(f'[Config] CubePart: {CUBEPART_CONFIG}')

CUBE3D_ENGINE = None
CUBEPART_PIPE = None


def free_engine():
    global CUBE3D_ENGINE, CUBEPART_PIPE
    CUBE3D_ENGINE = None
    CUBEPART_PIPE = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    import gc; gc.collect()
    print('[Engine] Both engines freed, CUDA cache cleared.')


def get_cube3d_engine():
    global CUBE3D_ENGINE
    if CUBE3D_ENGINE is not None:
        return CUBE3D_ENGINE
    from cube3d.inference.engine import EngineFast
    print('[Cube 3D] Loading EngineFast (CUDA only, ~16 GB VRAM) ...')
    t0 = time.time()
    CUBE3D_ENGINE = EngineFast(
        str(CUBE3D_CONFIG),
        str(CUBE3D_GPT),
        str(CUBE3D_VAE),
        device=torch.device('cuda'),
    )
    print(f'[Cube 3D] EngineFast ready in {time.time()-t0:.1f}s')
    return CUBE3D_ENGINE


def get_cubepart_pipeline():
    global CUBEPART_PIPE
    if CUBEPART_PIPE is not None:
        return CUBEPART_PIPE
    from cube_part.pipelines import PartShapeDenoiserPipeline
    print('[CubePart] Loading PartShapeDenoiserPipeline (~10 GB VRAM) ...')
    t0 = time.time()
    CUBEPART_PIPE = PartShapeDenoiserPipeline(
        config_path=str(CUBEPART_CONFIG),
        checkpoint_path=str(CUBEPART_DIT),
        vae_checkpoint_path=str(CUBEPART_VAE),
        device='cuda',
        extract_geometry_fn_name='extract_geometry_coarse_to_fine',
    )
    print(f'[CubePart] Pipeline ready in {time.time()-t0:.1f}s')
    return CUBEPART_PIPE


def gen_cube3d(
    prompt: str,
    use_bbox: bool,
    bbox_x: float,
    bbox_y: float,
    bbox_z: float,
    hi_res: bool,
    seed: int,
    progress=gr.Progress() if 'gr' in dir() else None,
):
    if not prompt or not prompt.strip():
        raise RuntimeError('Prompt is empty.')
    engine = get_cube3d_engine()
    try:
        from cube3d.inference.utils import normalize_bbox
    except ImportError:
        normalize_bbox = None
    bbox = [bbox_x, bbox_y, bbox_z] if use_bbox else None
    if bbox is not None and normalize_bbox is not None:
        bbox = normalize_bbox(bbox)
    if seed and int(seed) > 0:
        torch.manual_seed(int(seed))
    resolution_base = 9.0 if hi_res else 8.0
    t0 = time.time()
    mesh_v_f = engine.t2s(
        [prompt.strip()],
        use_kv_cache=True,
        resolution_base=resolution_base,
        bounding_box_xyz=bbox,
    )
    vertices, faces = mesh_v_f[0][0], mesh_v_f[0][1]
    # Postprocess: drop tiny disconnected components, simplify to 10% of original face count.
    try:
        from cube3d.mesh_utils.postprocessing import create_pymeshset, postprocess_mesh
        ms = create_pymeshset(vertices, faces)
        target_face_num = max(10_000, int(faces.shape[0] * 0.1))
        postprocess_mesh(ms, target_face_num)
        mesh = ms.current_mesh()
        vertices = mesh.vertex_matrix()
        faces = mesh.face_matrix()
        print(f'[Cube 3D] postprocessed: {faces.shape[0]} faces (target {target_face_num})')
    except Exception as e:
        print(f'[Cube 3D] postprocess skipped ({e})')
    elapsed = time.time() - t0
    out_path = OUT_DIR / 'cube3d_out.glb'
    trimesh.Trimesh(vertices=vertices, faces=faces).export(str(out_path))
    print(f'[Cube 3D] Saved to {out_path} in {elapsed:.1f}s')
    return str(out_path), f'✅ Generated in {elapsed:.1f}s — {faces.shape[0]} faces'


def gen_cubepart(
    mesh_path: str | None,
    parts_text: str,
    guidance_scale: float,
    num_inference_steps: int,
    resolution_base: float,
    num_samples: int,
    seed: int,
    progress=gr.Progress() if 'gr' in dir() else None,
):
    if not mesh_path or not Path(mesh_path).exists():
        raise RuntimeError('Please upload a .glb mesh.')
    parts = [p.strip() for p in parts_text.replace(',', '\n').splitlines() if p.strip()]
    if not parts:
        raise RuntimeError('Please enter at least one part name.')
    if len(parts) > 8:
        raise RuntimeError(f'Too many parts ({len(parts)}). Maximum is 8.')

    pipe = get_cubepart_pipeline()
    from cube_part.pipelines import ShapeInput
    from cube_part.utils.mesh import load_mesh, sample_surface
    import random
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed) if int(seed) > 0 else 0)
    t0 = time.time()
    mesh, _, _ = load_mesh(mesh_path)
    surface = sample_surface(mesh, num_samples=int(num_samples))
    surface = torch.from_numpy(surface).to(pipe.device).unsqueeze(0).float()
    latents, _ = pipe.encode_shape(surface)
    part_meshes = pipe.input_to_part_shape(
        ShapeInput(prompt=[parts], latents=latents),
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(num_inference_steps),
        resolution_base=float(resolution_base),
        seed=int(seed),
        timeshift=4.0,
        scheduler_type='dpm_solver',
    )
    # Color each part with a distinct color and assemble into a trimesh.Scene.
    import colorsys
    palette = []
    for i in range(max(len(parts), 1)):
        h = (i / max(len(parts), 1)) % 1.0
        r, g, b = colorsys.hsv_to_rgb(h, 0.55, 0.95)
        palette.append((int(r * 255), int(g * 255), int(b * 255)))
    palette = np.array(palette, dtype=np.uint8)
    scene = trimesh.Scene()
    kept = []
    for i, (vertices, faces) in enumerate(part_meshes):
        name = parts[i] if i < len(parts) else f'part_{i}'
        if vertices is None or faces is None or vertices.shape[0] == 0:
            kept.append((name, False))
            continue
        submesh = trimesh.Trimesh(vertices=vertices, faces=faces)
        submesh.visual.face_colors = palette[i % len(palette)]
        scene.add_geometry(submesh, geom_name=f'part_{i}_{name}')
        kept.append((name, True))
    if len(scene.geometry) == 0:
        raise RuntimeError('No parts produced any geometry.')
    out_path = OUT_DIR / 'cubepart_out.glb'
    scene.export(str(out_path))
    elapsed = time.time() - t0
    summary = ' | '.join(f"{'✓' if k else '✗'} {n}" for n, k in kept)
    print(f'[CubePart] Saved to {out_path} in {elapsed:.1f}s — {summary}')
    return str(out_path), f'✅ Generated {len(parts)} parts in {elapsed:.1f}s — {summary}'

import gradio as gr
print('[Core] Cube 3D + CubePart wrappers ready.')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with two tabs:
# @markdown - **Cube 3D (text → 3D)** — enter a prompt, get a single mesh
# @markdown - **CubePart (mesh → parts)** — upload a .glb, list part names, get per-part meshes

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

DEFAULT_PROMPTS_CUBE3D = [
    "A tall pagoda",
    "A pair of noise-canceling headphones",
    "Mechanical lobster with mechanical tank treads",
    "Lowpoly paper craft victorian rabbit",
    "A seashell",
]
DEFAULT_PARTS = (
    "body\n"
    "left front wheel\n"
    "right front wheel\n"
    "left rear wheel\n"
    "right rear wheel\n"
)

with gr.Blocks(title="Cube 3D + CubePart", theme=gr.themes.Soft(), fill_width=True) as demo:
    gr.Markdown(
        """
        # Roblox Cube 3D + CubePart
        Two text-/mesh-to-3D pipelines from the [Roblox Cube project](https://github.com/Roblox/cube).
        - **Cube 3D v0.5** — text → single mesh
        - **CubePart** — input mesh + parts schema → one mesh per part

        **Quick start:** Type a prompt or upload a mesh, hit **Generate**, watch the 3D preview update in the right panel.

        **VRAM notes:**
        - Cube 3D v0.5 `EngineFast`: ~16 GB VRAM (CUDA only)
        - CubePart: ~10 GB VRAM
        - Both loaded simultaneously: ~24 GB VRAM (L4 or A100)
        - Use the **VRAM** tab to free an engine if you get OOMs

        **Disclaimer:** Generated assets are not moderated by Roblox safety systems.
        """
    )

    with gr.Tabs():
        # ---------- Cube 3D v0.5 Tab ----------
        with gr.Tab("Cube 3D (text → 3D)"):
            with gr.Row():
                with gr.Column(scale=2):
                    c3d_prompt = gr.Textbox(
                        label="Text prompt",
                        value="A tall pagoda",
                        placeholder="Describe the object you want to generate...",
                        lines=2,
                    )
                    c3d_use_bbox = gr.Checkbox(
                        label="Use bounding-box conditioning (constrain aspect ratio)",
                        value=False,
                    )
                    with gr.Group() as c3d_bbox_group:
                        c3d_bbox_x = gr.Slider(0.1, 2.0, value=1.0, step=0.1,
                                               label="Length (X)", interactive=False,
                                               info="Target X extent of the generated mesh in unit cube space")
                        c3d_bbox_y = gr.Slider(0.1, 2.0, value=1.0, step=0.1,
                                               label="Height (Y)", interactive=False,
                                               info="Target Y extent. Tall objects: set this higher than X/Z")
                        c3d_bbox_z = gr.Slider(0.1, 2.0, value=1.0, step=0.1,
                                               label="Depth (Z)", interactive=False,
                                               info="Target Z extent. Deep objects: set this higher than X/Y")

                    def _toggle_bbox(on):
                        return gr.Slider(interactive=on), gr.Slider(interactive=on), gr.Slider(interactive=on)
                    c3d_use_bbox.change(
                        _toggle_bbox, inputs=[c3d_use_bbox],
                        outputs=[c3d_bbox_x, c3d_bbox_y, c3d_bbox_z],
                    )
                    with gr.Accordion("Advanced", open=False):
                        c3d_hi_res = gr.Checkbox(
                            label="Hi-res (resolution_base 9.0 vs 8.0)",
                            value=False,
                            info="Doubles the marching-cubes resolution — slower but more detail",
                        )
                        c3d_seed = gr.Number(value=0, label="Seed (0 = random)", precision=0,
                                              info="Fix the seed for reproducible generations")
                    c3d_btn = gr.Button("Generate (Cube 3D)", variant="primary")
                with gr.Column(scale=3):
                    c3d_output = gr.Model3D(
                        label="Generated mesh (.glb)",
                        clear_color=[0.92, 0.92, 0.94, 1.0],
                        height="45em",
                        interactive=False,
                    )
                    c3d_status = gr.Markdown("")
            gr.Examples(
                examples=[[p] for p in DEFAULT_PROMPTS_CUBE3D],
                inputs=[c3d_prompt],
                label="Try one of these prompts",
            )
            c3d_btn.click(
                gen_cube3d,
                inputs=[c3d_prompt, c3d_use_bbox, c3d_bbox_x, c3d_bbox_y, c3d_bbox_z, c3d_hi_res, c3d_seed],
                outputs=[c3d_output, c3d_status],
            )

        # ---------- CubePart Tab ----------
        with gr.Tab("CubePart (mesh → parts)"):
            gr.Markdown(
                """
                ### What CubePart does (and doesn't)

                - **Does**: takes an existing 3D mesh and **decomposes** it into semantic parts
                  (one mesh per part name you specify), with optional bounding-box conditioning.
                - **Does not**: generate meshes from text. That part (Stage 1 of the paper) is
                  not yet open-sourced — only the multi-part DiT (Stage 2) is released.
                - **Workaround**: run **Cube 3D** first to make a mesh, then come here to split it
                  into parts. Or click any thumbnail in the **Examples** gallery below.
                """
            )
            with gr.Row():
                with gr.Column(scale=2):
                    cp_mesh = gr.Model3D(
                        label="Input mesh (.glb, canonically aligned +Y up, +Z forward)",
                        clear_color=[0.92, 0.92, 0.94, 1.0],
                        height="28em",
                    )
                    cp_parts = gr.Textbox(
                        label="Part names (one per line, comma-separated, max 8)",
                        value=DEFAULT_PARTS,
                        lines=6,
                        info="Each line is a separate part the model will try to extract",
                    )
                    with gr.Accordion("Sampling settings", open=False):
                        cp_guidance = gr.Slider(
                            1.0, 15.0, value=7.5, step=0.1, label="Guidance scale",
                            info="Higher = stricter adherence to part names (default 7.5)",
                        )
                        cp_steps = gr.Slider(
                            1, 100, value=50, step=1, label="Diffusion steps",
                            info="More steps = finer detail, slower generation (default 50)",
                        )
                        cp_resolution = gr.Slider(
                            6.0, 10.0, value=8.5, step=0.5,
                            label="Marching-cubes log2 resolution",
                            info="8.5 default. Higher = denser mesh, slower",
                        )
                        cp_samples = gr.Slider(
                            16_000, 256_000, value=128_000, step=16_000,
                            label="Surface samples for encoding",
                            info="More samples = better input-mesh representation, slower",
                        )
                        cp_seed = gr.Number(value=42, precision=0, label="Seed")
                    cp_btn = gr.Button("Run CubePart", variant="primary")
                with gr.Column(scale=3):
                    cp_output = gr.Model3D(
                        label="Colored part scene (.glb)",
                        clear_color=[0.92, 0.92, 0.94, 1.0],
                        height="45em",
                        interactive=False,
                    )
                    cp_status = gr.Markdown("")

            # Examples gallery (matches the upstream HF Space UI). Click a
            # thumbnail to load that mesh + its parts list into the inputs above.
            gr.Markdown("### Examples — click a thumbnail to load it (these were pre-fetched in Step 2)")
            example_gal = gr.Gallery(
                value=[
                    (str(THUMB_DIR / f'{name}.png') if (THUMB_DIR / f'{name}.png').exists() else str(MESH_DIR / f'{name}.glb'),
                     name)
                    for name, _, _, _ in EXAMPLE_MESHES
                ],
                label=None, show_label=False,
                columns=5, object_fit='cover', height='auto',
                allow_preview=False,
                elem_id='cubepart_examples_gallery',
            )

            gr.Markdown(
                "**Tips:** Input mesh should be canonically aligned (+Y up, +Z forward). "
                "Watertight single-surface meshes work best. "
                "If your input is text-only, generate it first with the **Cube 3D** tab and feed the output here."
            )
            cp_btn.click(
                gen_cubepart,
                inputs=[cp_mesh, cp_parts, cp_guidance, cp_steps, cp_resolution, cp_samples, cp_seed],
                outputs=[cp_output, cp_status],
            )

            # Click handler for the gallery: load the picked mesh + its parts list
            def _on_example_select(evt: gr.SelectData):
                idx = int(evt.index) if evt.index is not None else 0
                idx = max(0, min(idx, len(EXAMPLE_MESHES) - 1))
                name, _, _, parts = EXAMPLE_MESHES[idx]
                mesh_path = str(MESH_DIR / f'{name}.glb')
                if not Path(mesh_path).exists():
                    return mesh_path, ''
                return mesh_path, '\n'.join(parts)
            example_gal.select(
                _on_example_select,
                outputs=[cp_mesh, cp_parts],
            )

        # ---------- VRAM Tab ----------
        with gr.Tab("VRAM"):
            gr.Markdown(
                """
                ### Free GPU memory
                Each engine uses ~10-16 GB VRAM. If you get OOMs, click below to free one
                (the next generation will re-load it on demand).
                """
            )
            with gr.Row():
                free_btn = gr.Button("Free all engines", variant="stop")
                free_status = gr.Markdown("")
            free_btn.click(lambda: ('[Engine] both engines freed, CUDA cache cleared.' if True else ''),
                          outputs=[free_status])
            free_btn.click(lambda: free_engine(), outputs=[], queue=False)

    gr.Markdown(
        """
        ---
        **License:** MIT (the CubePart / Cube 3D code; see [Roblox/cube](https://github.com/Roblox/cube)).
        The model weights are subject to the [OpenRAIL-M](https://huggingface.co/Roblox/cubepart#license) license — no high-risk downstream use.
        """
    )

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎲 Cube 3D + CubePart ready. Pick a tab and click Generate.", outputs=[c3d_status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run any time after Step 1.
import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')


In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Runs a single text-to-3D generation from the notebook. Useful for sanity-check after Step 1+2.
QUICK_PROMPT = "A friendly cartoon robot waving hello"
QUICK_USE_BBOX = False
QUICK_BBOX_X = 1.0
QUICK_BBOX_Y = 1.0
QUICK_BBOX_Z = 1.0
QUICK_HI_RES = False
QUICK_SEED = 0

if not QUICK_PROMPT or not QUICK_PROMPT.strip():
    raise SystemExit('QUICK_PROMPT is empty.')
out_path, status = gen_cube3d(
    QUICK_PROMPT, QUICK_USE_BBOX, QUICK_BBOX_X, QUICK_BBOX_Y, QUICK_BBOX_Z,
    QUICK_HI_RES, QUICK_SEED,
)
print(f'[QuickTest] {status}')
print(f'  path: {out_path}')
from IPython.display import FileLink, display
display(FileLink(out_path))

# Also feed the generated mesh into CubePart to test the decomposition pipeline end-to-end.
if QUICK_DO_CUBEPART and Path(out_path).exists():
    print('\n[QuickTest] Now feeding that mesh into CubePart for parts decomposition ...')
    try:
        cp_path, cp_status = gen_cubepart(
            mesh_path=out_path,
            parts_text=QUICK_PARTS,
            guidance_scale=7.5,
            num_inference_steps=50,
            resolution_base=8.5,
            num_samples=128_000,
            seed=42,
        )
        print(f'[QuickTest] {cp_status}')
        print(f'  path: {cp_path}')
        display(FileLink(cp_path))
    except Exception as e:
        print(f'[QuickTest] CubePart failed: {e}')


In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate one .glb per line of `BATCH_PROMPTS`. Saves to /content/cube3d_out/batch/.
BATCH_USE_BBOX = False
BATCH_BBOX_X = 1.0
BATCH_BBOX_Y = 1.0
BATCH_BBOX_Z = 1.0
BATCH_HI_RES = False
BATCH_SEED = 0

BATCH_PROMPTS = """\
A tall pagoda
A pair of noise-canceling headphones
A seashell
A cartoon cactus in a clay pot
A low-poly stylized tree with rounded leaves
"""

import os, time
batch_dir = OUT_DIR / 'batch'
batch_dir.mkdir(parents=True, exist_ok=True)
lines = [l.strip() for l in BATCH_PROMPTS.strip().splitlines() if l.strip()]
print(f'[Batch] {len(lines)} prompt(s) to generate.')
for i, line in enumerate(lines):
    print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
    try:
        out_path, status = gen_cube3d(
            line, BATCH_USE_BBOX, BATCH_BBOX_X, BATCH_BBOX_Y, BATCH_BBOX_Z,
            BATCH_HI_RES, BATCH_SEED,
        )
        safe = ''.join(c if c.isalnum() else '_' for c in line[:40]).strip('_') or f'item_{i:02d}'
        new_path = batch_dir / f'{i:02d}_{safe}.glb'
        os.rename(out_path, str(new_path))
        print(f'  -> {new_path}  ({status})')
    except Exception as e:
        print(f'  -> EXCEPTION: {e}')
        continue
print(f'\n[Done] {len(lines)} item(s) -> {batch_dir}')
